# Build Thumbnail Analysis Cache
Downloads thumbnails from Immich and builds the `thumbnails_analysis.json` cache from scratch.

For each photo, the cache stores:
- **IRQ metrics**: brightness, colorfulness, warmth, sky_score
- **Histograms**: 32-bin RGB density curves
- **Dominant colors**: 5-color KMeans palette (hex, rgb, percentage)

The process checkpoints every 200 photos so progress isn't lost on interruption.

In [1]:
import sys, os
sys.path.append(os.path.abspath('../backend/src'))

## Configuration
Set `LIMIT_COUNTRIES` to a list of country names to only cache photos from those countries.  
Leave as `None` or `[]` to cache **all** photos.

In [2]:
# ---- EDIT THIS ----
# Set to a list of countries to speed up caching, e.g. ["Italy", "France"]
# Set to None or [] to cache all photos
LIMIT_COUNTRIES:list = ["Indonesia"]

# Set True to force re-analysis (e.g. after algorithm changes)
FORCE_REBUILD = True

MAX_WORKERS = 8
CHECKPOINT_EVERY = 200

## 1. Load Photo IDs from Database

In [3]:
from infrastructure.clients.database import get_all_photos

df = get_all_photos()
print(f"Total photos in database: {len(df)}")

if LIMIT_COUNTRIES:
    df = df[df['country'].isin(LIMIT_COUNTRIES)].copy()
    print(f"Filtered to {len(df)} photos in: {', '.join(LIMIT_COUNTRIES)}")

all_ids = df['id'].tolist()
print(f"Photos to process: {len(all_ids)}")

if LIMIT_COUNTRIES:
    print(f"\nBreakdown by country:")
    print(df['country'].value_counts().to_string())

Total photos in database: 6848
Filtered to 676 photos in: Indonesia
Photos to process: 676

Breakdown by country:
country
Indonesia    676


## 2. Initialize Cache & Find Missing Entries

In [9]:
from infrastructure.persistence.cache import ThumbnailCache

CACHE_PATH = '../data/thumbnails_analysis.json'
cache = ThumbnailCache(CACHE_PATH)

missing_ids = cache.missing_from(all_ids)
print(f"Already cached: {len(cache)}")
print(f"Missing (need analysis): {len(missing_ids)}")

[cache] Loaded 1477 entries from ../data/thumbnails_analysis.json
Already cached: 1477
Missing (need analysis): 0


## 3. Download & Analyze Missing Photos
Uses multi-threaded downloads + OpenCV-based analysis with checkpointing.

In [10]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
from infrastructure.clients.immich import ImmichClient
from core.analysis.image_analyzer import analyze_thumbnail
from app.core.config import IMMICH_URL, API_KEY

client = ImmichClient(IMMICH_URL, API_KEY)

def process_one(asset_id: str):
    img = client.get_thumbnail(asset_id)
    if img is not None:
        return asset_id, analyze_thumbnail(img)
    return asset_id, None

if missing_ids:
    print(f"Analyzing {len(missing_ids)} photos ({MAX_WORKERS} threads)...")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = list(tqdm(
            executor.map(process_one, missing_ids),
            total=len(missing_ids),
            desc="Analyzing",
        ))
    
    success = 0
    failed = 0
    for asset_id, analysis in futures:
        if analysis:
            cache[asset_id] = analysis
            success += 1
        else:
            failed += 1
        # Checkpoint periodically
        if success > 0 and success % CHECKPOINT_EVERY == 0:
            cache.save()
            print(f"  Checkpoint: {success} saved")

    cache.save()
    print(f"\nDone! Analyzed {success} photos, {failed} failed.")
    print(f"Total cache size: {len(cache)}")
else:
    print("All photos already cached — nothing to do!")

All photos already cached — nothing to do!


## 4. Verify Cache Contents

In [11]:
import json

# Show cache stats
sample_id = next(iter(cache.data))


In [12]:
sample = cache[sample_id]

print(f"Cache entries: {len(cache)}")
print(f"\nSample entry keys: {list(sample.keys())}")
print(f"  brightness:  {sample['brightness']}")
print(f"  colorfulness: {sample['colorfulness']}")
print(f"  warmth:      {sample['warmth']}")
print(f"  sky_score:   {sample['sky_score']}")
print(f"  r_hist bins: {len(sample['r_hist'])}")

if 'dominant_colors' in sample:
    print(f"  dominant_colors: {len(sample['dominant_colors'])} colors")
    for c in sample['dominant_colors']:
        print(f"    {c['hex']}  ({c['proportion']:.1f}%)")
else:
    print("  ⚠ dominant_colors NOT found — run backfill (next cell)")

Cache entries: 1477

Sample entry keys: ['brightness', 'colorfulness', 'sky_score', 'warmth', 'r_hist', 'g_hist', 'b_hist', 'dominant_colors']
  brightness:  121.91
  colorfulness: 37.33
  warmth:      25.16
  sky_score:   -15.82
  r_hist bins: 32
  dominant_colors: 5 colors
    #7E745D  (42.9%)
    #66593D  (26.4%)
    #9C8F78  (21.9%)
    #413723  (5.9%)
    #A4C5D4  (2.8%)


## 5. (Optional) Backfill Dominant Colors
If you have old cache entries without `dominant_colors`, this cell adds them.

In [13]:
from core.analysis.image_analyzer import extract_dominant_colors

needs_backfill = [aid for aid, data in cache.data.items() if 'dominant_colors' not in data]
print(f"Entries needing backfill: {len(needs_backfill)}")

if needs_backfill:
    def backfill_one(asset_id):
        img = client.get_thumbnail(asset_id)
        if img is not None:
            return asset_id, extract_dominant_colors(img)
        return asset_id, None

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(
            executor.map(backfill_one, needs_backfill),
            total=len(needs_backfill),
            desc="Backfilling",
        ))

    updated = 0
    for asset_id, colors in results:
        if colors and asset_id in cache:
            cache[asset_id]['dominant_colors'] = colors
            updated += 1

    cache.save()
    print(f"Backfilled {updated} entries.")
else:
    print("All entries already have dominant_colors ✓")

Entries needing backfill: 0
All entries already have dominant_colors ✓
